# WakeWordProject — trening w Google Colab

Ten notebook uruchamia ten sam pipeline co lokalny `train.py`, **bez Dockera** (instalacja pip + klon **tego** repo + osobno `openWakeWord` v0.6.0 przez `colab_train.py` + konwersja TFLite).

**Zanim uruchomisz:**
- **Runtime → Change runtime type → GPU** (opcjonalnie; przy braku GPU trening i tak działa na CPU).
- **Git:** `REPO_URL` musi wskazywać na **WakeWordProject** (folder `colab/`, `scripts/`, `training_configs/`), np. [WakeWordProject](https://github.com/mrnz2/WakeWordProject) — **nie** samo upstreamowe `openWakeWord`. Klon trafia do `/content/WakeWordProject`. Repo **publiczne** albo dostęp zalogowanego konta w Colab.
- **Bez GitHuba:** `USE_GIT_CLONE = False`, wgraj ZIP projektu w **Files** → rozpakuj tak, by powstał katalog `/content/WakeWordProject` z podfolderami `colab/` i `scripts/` (np. `!unzip -q WakeWordProject.zip -d /content` jeśli ZIP zawiera jeden folder).
- **Zależności:** komórka z pakietami uruchamia `python colab/install_colab_deps.py` (torch z indeksu CUDA, potem `requirements-colab-train.txt`, potem `requirements-colab-tflite.txt`). **Nie** instalujemy `openwakeword` z PyPI — kod jest z klonu Git w `colab_train.py`.
- Komórka treningu ładuje `colab_train.py` **w tym samym procesie** co kernel (ten sam interpreter co po instalacji).
- Sesja Colab może się rozłączyć — warto **Drive** i `OUTPUT_DIR` na dysku.
- Przy **OOM** zmniejsz w YAML `tts_batch_size` / `n_samples`.
- **Błąd `git clone` exit 128:** zły URL, prywatne repo bez dostępu, albo gałąź `BRANCH` nie istnieje (spróbuj `master` zamiast `main`).

In [ ]:
# Opcjonalnie: Google Drive (wyniki w OUTPUT_DIR na dysku)
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import os
import shutil
import subprocess

# --- Ustawienia ---
USE_GIT_CLONE = True
# Musi to być repo **WakeWordProject** (nie sam openWakeWord) — inaczej zabraknie colab/, patchy, onnx2tf shim.
REPO_URL = "https://github.com/mrnz2/WakeWordProject.git"
BRANCH = "main"

# Docelowy katalog po klonie (jawnie podany w git clone — zawsze WakeWordProject)
PROJECT_ROOT = "/content/WakeWordProject"
OUTPUT_DIR = "/content/wakeword_outputs"
# Przykład na Drive (odkomentuj po zamontowaniu Drive):
# OUTPUT_DIR = "/content/drive/MyDrive/WakeWordProject_outputs"

if USE_GIT_CLONE:
    url = (REPO_URL or "").strip()
    if not url or "TWOJ_USER" in url or "YOUR_" in url.upper():
        raise ValueError(
            "Ustaw REPO_URL na prawdziwy adres HTTPS, np. "
            "https://github.com/twoj_login/WakeWordProject.git\n"
            "Albo: USE_GIT_CLONE = False i wgraj/rozpakuj projekt do "
            f"{PROJECT_ROOT} (foldery colab/, scripts/)."
        )
    if os.path.isdir(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT)
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, url, PROJECT_ROOT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(r.stderr or r.stdout)
        raise RuntimeError(
            f"git clone nie powiódł się (exit {r.returncode}). "
            "Sprawdź URL, dostęp do prywatnego repo oraz czy gałąź "
            f"'{BRANCH}' istnieje (czasem jest 'master' zamiast 'main')."
        ) from None
else:
    assert os.path.isdir(PROJECT_ROOT), (
        f"Brak {PROJECT_ROOT} — wgraj ZIP (Rozpakuj do WakeWordProject) lub włącz USE_GIT_CLONE."
    )

# Szybka weryfikacja: to nie upstream openWakeWord, tylko WakeWordProject z Colabem.
for _rel in ("colab/onnx_helper_shim.py", "scripts/run_onnx2tf_with_shim.py", "scripts/patch_openwakeword_train.py"):
    _p = os.path.join(PROJECT_ROOT, *_rel.split("/"))
    assert os.path.isfile(_p), f"Brak {_rel} — zły REPO_URL lub niepełny ZIP (oczekiwane repo WakeWordProject)."

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
%%capture
# libespeak-ng: espeak-phonemizer (piper-sample-generator). build-essential/python3-dev: ewentualna kompilacja webrtcvad.
!apt-get update -qq
!apt-get install -qq -y libespeak-ng-dev espeak-ng git build-essential python3-dev

In [ ]:
import os
import runpy
import sys

# Ten sam proces co kernel → cały log pip/ETAP jest w tej komórce (subprocess często „gubi” kontekst w Colab).
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/WakeWordProject"
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "/content/wakeword_outputs"
os.chdir(PROJECT_ROOT)

print("Python:", sys.version)
_inst = os.path.join(PROJECT_ROOT, "colab", "install_colab_deps.py")
# install_colab_deps przy błędzie rzuca RuntimeError z ETAP + końcówką logu pip (widać w tracebacku).
runpy.run_path(_inst, run_name="__main__")

import torch

print("OK — interpreter:", sys.executable)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/WakeWordProject"
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "/content/wakeword_outputs"

# Ten sam interpreter co kernel (unika sytuacji: subprocess = /usr/bin/python3 bez torch,
# a pip dołożył pakiety do innego Pythona).
os.chdir(PROJECT_ROOT)
_script = Path(PROJECT_ROOT) / "colab" / "colab_train.py"
_argv = [
    str(_script),
    "--project_root",
    str(PROJECT_ROOT),
    "--output_dir",
    str(OUTPUT_DIR),
    "--training_config",
    "training_configs/hey_lolita_colab.yml",
]

_old_argv = sys.argv[:]
try:
    sys.argv = _argv
    spec = importlib.util.spec_from_file_location("_colab_train_main", _script)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Nie wczytano: {_script}")
    _mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(_mod)
    _mod.main()
finally:
    sys.argv = _old_argv

print("Gotowe. Interpreter:", sys.executable)

# Ponowny sam trening (clipy już w OUTPUT_DIR) — zmień _argv, np. dopisz:
# "--skip_setup", "--train_only"
# Wymuszenie: "--force_overwrite" | Tylko ONNX: "--skip_tflite"

## Wyniki

W `OUTPUT_DIR` powinny pojawić się m.in. `hey_lolita.onnx` i `hey_lolita.tflite`. Pobierz je (**Files** w panelu Colab) albo skopiuj na Drive.

Pełniejsza jakość (więcej próbek / kroków): skopiuj parametry z `training_configs/hey_lolita.yml` do pliku colab lub użyj `--training_config training_configs/hey_lolita.yml` (dłużej, większe ryzyko timeoutu i OOM).